In [32]:
import os
import zipfile
import numpy as np
import pandas as pd

# Disabilita i warning di Pandas legati alla frammentazione della memoria
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# 1. Configurazione dei percorsi dei file
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))

zip_path = os.path.join(base_dir, "data", "raw", "archive.zip")
extract_path = os.path.join(base_dir, "data", "raw", "extracted_football_data")
output_processed_path = os.path.join(base_dir, "data", "processed", "df_tattics_processed.parquet")

# 2. Processo di estrazione dati dal dataset (.zip)
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"[ERRORE] Impossibile trovare il file ZIP in: {zip_path}\n"
        f"Verifica il nome del file all'interno di data/raw/"
    )

# Controlla se la cartella esiste già ed è popolata per evitare l'unzip inutile
files_da_caricare = ['combined_matches.csv', 'combined_matches_2068 2269 2417.csv']
gia_estratto = os.path.exists(extract_path) and all(os.path.exists(os.path.join(extract_path, f)) for f in files_da_caricare)

if gia_estratto:
    print(f"[INFO] I file CSV sono già presenti in: {extract_path}. Salto lo scompattamento.")
else:
    print(f"[INFO] Scompattamento del dataset in corso da: {zip_path}...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"[OK] Estrazione completata!")

print("Contenuto directory:", os.listdir(extract_path))

# 3. Caricamento e unificazione dei file .csv
# Definisce i file attesi all'interno della cartella scompattata
files_da_caricare = ['combined_matches.csv', 'combined_matches_2068 2269 2417.csv']

dfs = []
# Seleziona solo le colonne strettamente necessarie
colonne_utili = [
    'match_id', 'frame_number', 'period', 'timestamp_seconds',
    'player_id', 'team_name', 'team_type', 'x', 'y', 'is_goalkeeper'
]

for file in files_da_caricare:
    full_path = os.path.join(extract_path, file)
    if os.path.exists(full_path):
        print(f"[INFO] Caricamento selettivo in corso per: {file}...")
        dfs.append(pd.read_csv(full_path, usecols=colonne_utili))
    else:
        print(f"[AVVISO] File opzionale non trovato: {file}. Salto il caricamento.")

if not dfs:
    raise FileNotFoundError("[ERRORE] Nessun file CSV valido è stato caricato. Verifica il contenuto dello ZIP.")

# Unisce i blocchi di partite reali caricati
df_kaggle = pd.concat(dfs, ignore_index=True)
print(f"-> Dataset unificato con successo. Totale righe grezze caricate: {len(df_kaggle)}")


# 4. Pulizia e calcolo dei centroidi
print("\n[INFO] Avvio pulizia dati e isolamento dei giocatori di movimento...")

# Gestisce la colonna is_goalkeeper convertendola in booleano (evita stringhe sporche)
df_kaggle['is_goalkeeper'] = df_kaggle['is_goalkeeper'].astype(str).str.lower().str.contains('true|1') # type: ignore

# Filtra via i portieri ed elimina i record con coordinate (X;Y) mancanti (palla fuori/tempi morti)
df_outfield = df_kaggle[df_kaggle['is_goalkeeper'] == False].dropna(subset=['x', 'y']).copy() # type: ignore

print("[INFO] Calcolo cinematica: estrazione dei baricentri istantanei di squadra...")
# Utilizza .transform('mean') per calcolare il baricentro istante per istante mantenendo l'indice originario
centroide = df_outfield.groupby(['match_id', 'frame_number', 'team_name'])[['x', 'y']].transform('mean')

# Calcola le coordinate relative riferite al centroide dinamico (fondamentale per KMeans)
df_outfield['x_rel'] = df_outfield['x'] - centroide['x']
df_outfield['y_rel'] = df_outfield['y'] - centroide['y']

# Crea dei micro-chunk temporali da 5 minuti (300 secondi) per l'analisi
df_outfield['chunk_id'] = (df_outfield['timestamp_seconds'] // 300).astype(int)

print(f"\n=== PRE-PROCESSING COMPLETATO: {len(df_outfield)} RIGHE IN OPEN PLAY PRONTE ===")


# 5. Salvataggio serializzato ed anteprima risultati
# Salva in formato Parquet per mantenere i tipi di dato intatti ed ottimizzare lo spazio
os.makedirs(os.path.dirname(output_processed_path), exist_ok=True)
df_outfield.to_parquet(output_processed_path, index=False)
print(f"[OK] Dataset processato salvato correttamente in: {output_processed_path}")

print("\n=== ESEMPIO DI OUTPUT GEOMETRICO GENERATO ===")
print(df_outfield.head(3).to_string())

[INFO] I file CSV sono già presenti in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\data\raw\extracted_football_data. Salto lo scompattamento.
Contenuto directory: ['combined_matches.csv', 'combined_matches_2068 2269 2417.csv']
[INFO] Caricamento selettivo in corso per: combined_matches.csv...
[INFO] Caricamento selettivo in corso per: combined_matches_2068 2269 2417.csv...
-> Dataset unificato con successo. Totale righe grezze caricate: 6453019

[INFO] Avvio pulizia dati e isolamento dei giocatori di movimento...
[INFO] Calcolo cinematica: estrazione dei baricentri istantanei di squadra...

=== PRE-PROCESSING COMPLETATO: 6298277 RIGHE IN OPEN PLAY PRONTE ===
[OK] Dataset processato salvato correttamente in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\data\processed\df_tattics_processed.parquet

=== ESEMPIO DI OUTPUT GEOMETRICO GENERATO ===
   match_id  frame_number  period  timestamp_seconds  player_id       team_name team_type          x         y  is_goalke

In [34]:
import os
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from mplsoccer import Pitch
from sklearn.cluster import AgglomerativeClustering, KMeans

# Lista globale in cui verranno salvati i tuoi dati validati
dataset_addestramento_finale = []

# Raggruppa per match, squadra e blocco da 5 minuti
gruppi_chunk = df_outfield.groupby(["match_id", "team_name", "chunk_id"])
iteratore_gruppi = iter(gruppi_chunk)

# Area di output dedicata e permanente per i grafici
area_grafico = widgets.Output()

centri_ruoli = None
modulo_proposto = "N/A"

# 1. Inizializzazione statica dell'interfaccia
# Dichiara i componenti una volta sola per evitare il freeze del testo sui pulsanti
btn_accetta = widgets.Button(description="Accetta Proposta", button_style="success", layout=widgets.Layout(width="240px"))
btn_scarta = widgets.Button(description="Scarta questo Blocco", button_style="warning", layout=widgets.Layout(width="160px"))
btn_esci = widgets.Button(description="SALVA ED ESCI", button_style="danger", layout=widgets.Layout(width="150px"))

txt_personalizzato = widgets.Text(value="", placeholder="Esempio: 4-2-3-1, 3-4-1-2", description="Modulo Tuo:", layout=widgets.Layout(width="300px"))
btn_invia_custom = widgets.Button(description="Invia Modulo Personalizzato", button_style="info", layout=widgets.Layout(width="210px"))

btn_442 = widgets.Button(description="4-4-2", layout=widgets.Layout(width="90px"))
btn_433 = widgets.Button(description="4-3-3", layout=widgets.Layout(width="90px"))
btn_352 = widgets.Button(description="3-5-2", layout=widgets.Layout(width="90px"))

# Strutturazione dei box contenitori stabili
box_azioni_rapide = widgets.HBox([btn_accetta, btn_scarta, btn_esci])
box_input_libero = widgets.HBox([txt_personalizzato, btn_invia_custom])
box_scorciatoie = widgets.HBox([widgets.Label("Scorciatoie veloci:"), btn_442, btn_433, btn_352])

interfaccia_completa = widgets.VBox([
    area_grafico,
    widgets.HTML("<br>"),
    box_azioni_rapide,
    widgets.HTML("<br>"),
    box_input_libero,
    widgets.HTML("<br>"),
    box_scorciatoie
])



# 2. Logica di business e aggiornamento
def elabora_prossimo_blocco():
    global centri_ruoli, modulo_proposto, iteratore_gruppi

    centri_ruoli = None
    modulo_proposto = "N/A"

    try:
        (match_id, team_name, chunk_id), frame_chunk = next(iteratore_gruppi)
    except StopIteration:
        with area_grafico:
            clear_output()
        clear_output()
        print("=== COMPLIMENTI: TUTTI I BLOCCHI DEL DATASET SONO STATI ANALIZZATI ===")
        compila_dataset_finale()
        return

    # Filtro di stabilità sulle sostituzioni
    giocatori_unici = frame_chunk["player_id"].nunique()
    if giocatori_unici < 10 or len(frame_chunk) < 200:
        elabora_prossimo_blocco()
        return

    top_10_players = frame_chunk["player_id"].value_counts().head(10).index
    df_filtrato = frame_chunk[frame_chunk["player_id"].isin(top_10_players)]
    X_valori = df_filtrato[["x_rel", "y_rel"]].values

    # Calcolo della proposta automatica
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans.fit(X_valori)
    centri_ruoli = kmeans.cluster_centers_

    centri_ruoli = centri_ruoli[np.argsort(centri_ruoli[:, 0])]
    altezze_x = centri_ruoli[:, 0].reshape(-1, 1)

    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=7.5, metric="euclidean", linkage="complete"
    )
    labels_grezze = clustering.fit_predict(altezze_x)
    num_reparti = len(np.unique(labels_grezze))

    if num_reparti < 3 or num_reparti > 4:
        km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(altezze_x)
        km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(altezze_x)
        from sklearn.metrics import silhouette_score

        if silhouette_score(altezze_x, km3.labels_) >= silhouette_score(altezze_x, km4.labels_):
            labels_grezze = km3.labels_
            num_reparti = 3
        else:
            labels_grezze = km4.labels_
            num_reparti = 4

    centri_cluster_x = [np.mean(altezze_x[labels_grezze == r]) for r in range(num_reparti)]
    ordine_reparti = np.argsort(centri_cluster_x)
    mappa_ordine = {vecchio: nuovo for nuovo, vecchio in enumerate(ordine_reparti)}
    labels_reparti = np.array([mappa_ordine[l] for l in labels_grezze])

    modulo_proposto = "-".join([str(np.sum(labels_reparti == r)) for r in range(num_reparti)])

    # Aggiorna dinamicamente la scritta sul pulsante e riabilita i controlli
    btn_accetta.description = f"Accetta Proposta ({modulo_proposto})"
    btn_accetta.disabled = False
    btn_scarta.disabled = False
    txt_personalizzato.value = ""  # Resetta il campo di testo libero per il nuovo turno

    # Rendering grafico della lavagnetta in outpu (non file salvato)
    with area_grafico:
        clear_output(wait=True)
        PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0
        pitch = Pitch(
            pitch_type="custom", pitch_length=PITCH_LENGTH, pitch_width=PITCH_WIDTH,
            pitch_color="#22312b", line_color="#c7d5cc"
        )
        fig, ax = pitch.draw(figsize=(7, 4.5))
        fig.patch.set_facecolor("#22312b")

        x_plot = centri_ruoli[:, 0] + (PITCH_LENGTH / 2)
        y_plot = centri_ruoli[:, 1] + (PITCH_WIDTH / 2)

        for r in range(num_reparti):
            mask = labels_reparti == r
            if np.sum(mask) > 1:
                idx_y = np.argsort(y_plot[mask])
                ax.plot(
                    x_plot[mask][idx_y], y_plot[mask][idx_y],
                    color="#e63946", linestyle="-", linewidth=2, alpha=0.6
                )

        pitch.scatter(x_plot, y_plot, ax=ax, color="#e63946", edgecolors="#ffffff", s=220, zorder=3)

        for num_progressivo, idx in enumerate(np.argsort(x_plot), start=1):
            ax.text(
                x_plot[idx], y_plot[idx], str(num_progressivo),
                color="#ffffff", fontsize=8, ha="center", va="center", weight="bold"
            )

        ax.set_title(
            f"Match: {match_id} | Team: {team_name} | Mins {chunk_id*5}-{(chunk_id+1)*5}\nPROPOSTA CALCOLATA: {modulo_proposto}",
            color="#c7d5cc", fontsize=10, weight="bold"
        )
        plt.show()

    clear_output(wait=True)
    print(f"[INPUT] Valida blocco corrente per: {team_name}")
    display(interfaccia_completa)



# 3. Gestione degli eventi di selezione
def al_clic(b):
    # Protezione anti-spam
    btn_accetta.disabled = True
    btn_scarta.disabled = True

    if b.description.startswith("Accetta"):
        salva_frame(modulo_proposto)
    elif b.description.startswith("Scarta"):
        print("[INFO] Blocco ignorato.")
    elif b.description.startswith("SALVA"):
        with area_grafico:
            clear_output()
        clear_output()
        compila_dataset_finale()
        return
    elif b.description.startswith("Invia Modulo") or b.description == "Invia Correzione":
        modulo_scritto = txt_personalizzato.value.strip()
        if modulo_scritto == "":
            btn_accetta.disabled = False
            btn_scarta.disabled = False
            return
        salva_frame(modulo_scritto)
    else:
        # Gestisce i bottoni rapidi statici (4-4-2, 4-3-3, 3-5-2)
        salva_frame(b.description)

    # Avanza al blocco successivo caricando la nuova descrizione
    elabora_prossimo_blocco()


# Collega gli elementi al gestore una volta sola
for elemento in [btn_accetta, btn_scarta, btn_esci, btn_invia_custom, btn_442, btn_433, btn_352]:
    elemento.on_click(al_clic)


from typing import cast  # Metti questo import in cima al file se vuoi, o lascialo qui


def salva_frame(modulo_risolto):
    _centri_temp = globals().get('centri_ruoli', None)
    if _centri_temp is not None and isinstance(_centri_temp, np.ndarray):
        centri_validi = cast(np.ndarray, _centri_temp)
        centri_ordinati_x = centri_validi[:, 0]
        centri_ordinati_y = centri_validi[:, 1]
        vettore_riga = centri_ordinati_x.tolist() + centri_ordinati_y.tolist() + [modulo_risolto]
        dataset_addestramento_finale.append(vettore_riga)

def compila_dataset_finale():
    global df_training_reale
    colonne_dataset = [f"x_{idx}" for idx in range(10)] + [f"y_{idx}" for idx in range(10)] + ["target_modulo"]

    df_training_reale = pd.DataFrame(dataset_addestramento_finale, columns=colonne_dataset)
    print(f"=== GROUND TRUTH COSTRUITA CON SUCCESSO ===")
    print(f"Hai validato e corretto {len(df_training_reale)} blocchi tattici reali per il Random Forest.")
    if not df_training_reale.empty:
        display(df_training_reale.head())

# Avvia il loop del framework tattico
print("Inizializzazione interfaccia flessibile... Attendi il caricamento del primo blocco.")
elabora_prossimo_blocco()

In [35]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans

# Recupera i dataset e i modelli in modo pulito dallo spazio globale
_df_temp = globals().get('df_training_reale', None)
df_training_reale = _df_temp if isinstance(_df_temp, pd.DataFrame) else None

_df_tattico_temp = globals().get('df_tattico', None)
df_tattico = _df_tattico_temp if isinstance(_df_tattico_temp, pd.DataFrame) else None

# Nome del modello che viene sviluppato
analista = globals().get('analista', None)

# 1. Addestramento random forest classifier
if df_training_reale is not None and not df_training_reale.empty:
    if df_training_reale['target_modulo'].nunique() > 1:

        # Filtro di protezione anti-membro unico
        conteggi = df_training_reale['target_modulo'].value_counts()
        # Tiene solo i moduli che compaiono almeno 2 volte per permettere lo split stratificato
        moduli_validi = conteggi[conteggi >= 2].index

        df_filtrato_training = df_training_reale[df_training_reale['target_modulo'].isin(moduli_validi)].copy()

        if df_filtrato_training['target_modulo'].nunique() < 2:
            print("[ATTENZIONE] Dopo il filtro antigremlins mancano abbastanza classi distinte con almeno 2 esempi.")
            print("Torna alla cella dell'interfaccia interattiva e valida più moduli con lo stesso nome!")
        else:
            # Converte esplicitamente in array NumPy puri per evitare conflitti con PyArrow
            X = df_filtrato_training.drop(columns=['target_modulo']).to_numpy(dtype=np.float64)
            y = df_filtrato_training['target_modulo'].to_numpy(dtype=object)

            # Suddivisione con stratificazione sicura al 100%
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

            print(f"[MACHINE LEARNING] Addestramento del Random Forest su {len(df_filtrato_training)} blocchi purificati...")
            analista = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42)
            analista.fit(X_train, y_train)

            print("\n" + "=" * 60)
            print("=== REPORT DI VALIDAZIONE SCIENTIFICA DEL MODELLO ===")
            print("=" * 60)
            y_pred = analista.predict(X_test)
            print(classification_report(y_test, y_pred, zero_division=0))
    else:
        print("[ATTENZIONE] Il dataset ha un solo modulo unico registrato. Serve almeno un'altra variante tattica per addestrare il modello!")
else:
    print("[ATTENZIONE] Dataset di training vuoto o insufficiente. Valida qualche modulo nell'interfaccia widgets!")


# 2. Match analysis
# Verifica se sia il dataset che il modello addestrato sono pronti
if df_tattico is not None and 'analista' in locals():
    print("\n" + "=" * 80)
    print("[INFO] Applicazione del modello addestrato su dati Kaggle alla partita di nostro interesse")
    print("=" * 80)

    try:
        # 1. Isola un chunk di prova (es. i primi record stabili della squadra H o A)
        # Prende un sottoinsieme temporale della squadra di casa ("H")
        df_match_test = df_tattico[df_tattico["team"] == "H"].head(500)

        if len(df_match_test) >= 110:
            # 2. Calcola i 10 centroidi spaziali stabili
            X_posizioni = df_match_test[["x_rel", "y_rel"]].to_numpy(dtype=np.float64)
            kmeans_match = KMeans(n_clusters=10, random_state=42, n_init=10)
            kmeans_match.fit(X_posizioni)
            centri_match = kmeans_match.cluster_centers_

            # 3. Ordinamento rigido sulla profondità (asse X) per allinearlo alle 20 colonne del modello
            centri_match = centri_match[np.argsort(centri_match[:, 0])]

            # 4. Srotola la matrice (10, 2) in un vettore piatto riga (1, 20) -> [x0..x9, y0..y9]
            X_nuovo_match = np.hstack([centri_match[:, 0], centri_match[:, 1]]).reshape(1, -1)

            # 5. Esegue la classificazione automatica senza regole pre-impostate
            modulo_predetto = analista.predict(X_nuovo_match)

            print(f"\n[RISULTATO INFERENZA TATTICA]")
            print(f"       -> Sistema rilevato automaticamente per H: {modulo_predetto[0]}")
            print("\n[OK] Il modello è pronto per generare la timeline dei moduli del match reale!")
            print("Puoi mappare l'intero df_tattico ciclando sui chunk usando 'analista.predict(X_nuovo)'.")
        else:
            print("[AVVISO] df_tattico presente ma record insufficienti nel segmento di test per estrarre 10 ruoli.")

    except Exception as e:
        print(f"[ERRORE INFERENZA] Impossibile eseguire la predizione di test sul match: {e}")

elif 'analista' in locals():
    print("\n[INFO] Pipeline completata con successo.")

[MACHINE LEARNING] Addestramento del Random Forest su 184 blocchi purificati...

=== REPORT DI VALIDAZIONE SCIENTIFICA DEL MODELLO ===
              precision    recall  f1-score   support

       1-4-5       0.50      1.00      0.67         1
       1-5-4       1.00      0.50      0.67         2
       2-3-5       0.00      0.00      0.00         1
       2-4-4       0.43      0.75      0.55         4
       2-5-3       0.67      0.67      0.67         3
     3-1-4-2       0.00      0.00      0.00         0
       3-3-4       0.33      0.33      0.33         3
       3-4-3       1.00      0.60      0.75         5
       3-5-2       0.33      0.50      0.40         2
     4-1-4-1       0.00      0.00      0.00         0
       4-2-4       0.00      0.00      0.00         2
       4-3-3       0.00      0.00      0.00         3
       4-4-2       0.50      0.80      0.62         5
       4-5-1       0.00      0.00      0.00         1
       5-1-4       0.00      0.00      0.00         1


In [36]:
import json
import os
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Forza a importare la funzione dal file utils_graphics2.py
try:
    from utils_graphics2 import visualizza_lavagnetta_ungherese2
    print("[OK] Modulo 'utils_graphics' importato correttamente.")
except ImportError:
    raise ImportError(
        "[ERRORE CRITICO] Impossibile trovare 'utils_graphics2.py'. "
        "Assicurati che il file sia nella stessa directory di questo notebook su IntelliJ."
    )

# 1. Configurazione dei percorsi dei file
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
base_project_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))

tracking_parquet_path = os.path.join(base_project_dir, "data", "processed", "integrated_tracking_data.parquet")
opta_events_json_path = os.path.join(base_project_dir, "data", "raw", "MA3_opta_matchevent.json")


# 2. Caricamento dari di tracking OPTA e sincornizzazione
print("[INFO] Ricaricamento e sincronizzazione dei file posizionali in corso...")
if not os.path.exists(opta_events_json_path):
    raise FileNotFoundError(f"[ERRORE] File JSON Opta non trovato in: {opta_events_json_path}")

with open(opta_events_json_path, "r", encoding="utf-8") as f:
    eventi_data = json.load(f)

events_list = eventi_data["liveData"]["event"]
parsed_events = [
    {
        "period_id": ev["periodId"],
        "timestamp_ms": int(
            datetime.fromisoformat(ev["timeStamp"].replace("Z", "")).timestamp()
            * 1000
        ),
    }
    for ev in events_list
]
df_events = (
    pd.DataFrame(parsed_events)
    .sort_values("timestamp_ms")
    .reset_index(drop=True)
)

t1_start = df_events[df_events["period_id"] == 1]["timestamp_ms"].min()
t1_end = df_events[df_events["period_id"] == 1]["timestamp_ms"].max()
t2_start = df_events[df_events["period_id"] == 2]["timestamp_ms"].min()
t2_end = df_events[df_events["period_id"] == 2]["timestamp_ms"].max()

if not os.path.exists(tracking_parquet_path):
    raise FileNotFoundError(f"[ERRORE] File Parquet Tracking non trovato in: {tracking_parquet_path}")

df_tracking = pd.read_parquet(tracking_parquet_path)
df_tracking["periodo"] = 0
df_tracking.loc[
    (df_tracking["timestamp"] >= t1_start)
    & (df_tracking["timestamp"] <= t1_end),
    "periodo",
] = 1
df_tracking.loc[
    (df_tracking["timestamp"] >= t2_start)
    & (df_tracking["timestamp"] <= t2_end),
    "periodo",
] = 2
df_tracking = df_tracking[df_tracking["periodo"].isin([1, 2])].copy()

# Inversione di campo secondo tempo
df_tracking.loc[df_tracking["periodo"] == 2, ["x", "y"]] = -df_tracking.loc[
    df_tracking["periodo"] == 2, ["x", "y"]
]

# Identificazione Portieri
id_portieri = (
    df_tracking.groupby(["team", "player_id"])["x"].mean().reset_index()
)
id_gk_H = (
    id_portieri[id_portieri["team"] == "H"]
    .sort_values(by="x")
    .iloc[0]["player_id"]
)
id_gk_A = (
    id_portieri[id_portieri["team"] == "A"]
    .sort_values(by="x", ascending=False)
    .iloc[0]["player_id"]
)


def prepara_dati_scientifici(df_track):
    X_MIN, X_MAX = -52.5, 52.5
    Y_MIN, Y_MAX = -34.0, 34.0
    PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0
    df_clean = df_track.dropna(subset=["x", "y"]).copy()

    x_clamped = np.clip(df_clean["x"], X_MIN, X_MAX)
    y_clamped = np.clip(df_clean["y"], Y_MIN, Y_MAX)

    df_clean["x_norm"] = ((x_clamped - X_MIN) / (X_MAX - X_MIN)) * PITCH_LENGTH
    df_clean["y_norm"] = ((y_clamped - Y_MIN) / (Y_MAX - Y_MIN)) * PITCH_WIDTH

    is_not_gk = (df_clean["player_id"] != id_gk_H) & (
            df_clean["player_id"] != id_gk_A
    )
    centroide_frame = (
        df_clean[is_not_gk]
        .groupby(["timestamp", "team"])[["x_norm", "y_norm"]]
        .mean()
        .reset_index()
    )
    centroide_frame.rename(
        columns={"x_norm": "cx", "y_norm": "cy"}, inplace=True
    )

    df_clean = pd.merge(
        df_clean, centroide_frame, on=["timestamp", "team"], how="left"
    )
    df_clean["x_rel"] = df_clean["x_norm"] - df_clean["cx"]
    df_clean["y_rel"] = df_clean["y_norm"] - df_clean["cy"]
    return df_clean.dropna(subset=["x_rel", "y_rel"])


df_tattico_locale = prepara_dati_scientifici(df_tracking)
df_tattico_locale = df_tattico_locale[
    (df_tattico_locale["player_id"] != id_gk_H)
    & (df_tattico_locale["player_id"] != id_gk_A)
    ].copy()


# 3. Interfaccia random forest con proposte
print(
    "\n[INFO] Configurazione motore di Match Analysis basato su Random Forest..."
)

# Sincronizzazione sicura del modello addestrato nella cella precedente su IntelliJ
analista = globals().get('analista', None)

def individua_modulo_con_random_forest(df_finestra_squadra):
    if analista is None:
        print("[ERRORE] Il modello 'analista' non è in memoria. Addestralo nella cella precedente!")
        return "N/A", None, None

    if len(df_finestra_squadra) < 200:
        return "N/A", None, None

    # Risoluzione a bug dati dalle sostituzioni
    giocatori_dominanti = (
        df_finestra_squadra["player_id"].value_counts().head(10).index
    )
    df_filtrato_10 = df_finestra_squadra[
        df_finestra_squadra["player_id"].isin(giocatori_dominanti)
    ]

    X_valori = df_filtrato_10[["x_rel", "y_rel"]].values
    if len(X_valori) < 100:
        return "N/A", None, None

    kmeans_ruoli = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans_ruoli.fit(X_valori)
    centri_ruoli_ideali = kmeans_ruoli.cluster_centers_

    idx_ordine_tattico = np.argsort(centri_ruoli_ideali[:, 0])
    centri_ruoli_ideali = centri_ruoli_ideali[idx_ordine_tattico]

    vettore_feature = list(centri_ruoli_ideali[:, 0]) + list(
        centri_ruoli_ideali[:, 1]
    )

    # Chiamata al modello addestrato su dataset
    modulo_predetto_ia = analista.predict([vettore_feature])[0]

    try:
        # FIX REFUSO: Usiamo 'modulo_predetto_ia' (il nome aggiornato della variabile locale)
        parti_modulo = [int(x) for x in modulo_predetto_ia.split("-")]
        labels_reparti = []
        for reparto_idx, numero_giocatori in enumerate(parti_modulo):
            labels_reparti.extend([reparto_idx] * numero_giocatori)
        labels_reparti = np.array(labels_reparti)
    except (ValueError, AttributeError):
        labels_reparti = np.zeros(10)

    # Allineato anche il return finale
    return modulo_predetto_ia, centri_ruoli_ideali, labels_reparti


# 4. Pipeline di analisi ogni 15 minuti di gioco
unique_timestamps = sorted(df_tattico_locale["timestamp"].unique())
chunks = np.array_split(unique_timestamps, 6)

# Nome della cartella in cui verranno salvate le predizioni
NOME_CARTELLA_DESTINAZIONE = "tactics03"

print("\n=== AVVIO ESTRAZIONE GRAFICA: PIPELINE CON RANDOM FOREST CLASSIFIER ===")

for i, chunk in enumerate(chunks, start=1):
    df_chunk = df_tattico_locale[df_tattico_locale["timestamp"].isin(chunk)]
    testo_fase = (
        f"Fase di gioco {i} (Minuti indicativi: {int((i-1)*15)}' - {int(i*15)}')"
    )

    print(f"\n" + "=" * 80)
    print(f"[ELABORAZIONE CHUNK - CLASSIFICATORE IA] {testo_fase}")
    print("=" * 80)

    for team_code in ["H", "A"]:
        nome_team = "H (Casa)" if team_code == "H" else "A (Ospite)"
        df_team_chunk = df_chunk[df_chunk["team"] == team_code]

        modulo, centri, labels = individua_modulo_con_random_forest(
            df_team_chunk
        )

        if modulo == "N/A" or centri is None or labels is None:
            print(
                f"       [AVVISO] Dati insufficienti per {nome_team} nella Fase {i}. Salto la predizione."
            )
            continue

        print(f"       -> {nome_team} | Modulo Predetto dall'IA: {modulo}")

        # Passa il parametro del nome della directory come nome_script
        # Analisi salvate in out/MLAnalysis/tactics03/
        visualizza_lavagnetta_ungherese2(
            nome_team, team_code, testo_fase, modulo, centri, labels,
            nome_script=NOME_CARTELLA_DESTINAZIONE
        )

print("\n=== PIPELINE MACHINE LEARNING COMPLETATA CON SUCCESSO ===")

[OK] Modulo 'utils_graphics' importato correttamente.
[INFO] Ricaricamento e sincronizzazione dei file posizionali in corso...

[INFO] Configurazione motore di Match Analysis basato su Random Forest...

=== AVVIO ESTRAZIONE GRAFICA: PIPELINE CON RANDOM FOREST CLASSIFIER ===

[ELABORAZIONE CHUNK - CLASSIFICATORE IA] Fase di gioco 1 (Minuti indicativi: 0' - 15')
       -> H (Casa) | Modulo Predetto dall'IA: 4-4-2
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\MLAnalysis\tactics03\H_(Casa)_Fase_di_gioco_1.png
       -> A (Ospite) | Modulo Predetto dall'IA: 4-4-2
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\MLAnalysis\tactics03\A_(Ospite)_Fase_di_gioco_1.png

[ELABORAZIONE CHUNK - CLASSIFICATORE IA] Fase di gioco 2 (Minuti indicativi: 15' - 30')
       -> H (Casa) | Modulo Predetto dall'IA: 3-5-2
       [GRAFICO OMNI] Lavagnetta salvata con succe

In [37]:
import json
import os
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Forza a importare la funzione dal file utils_graphics2.py
try:
    from utils_graphics2 import visualizza_lavagnetta_ungherese2
    print("[OK] Modulo 'utils_graphics2' importato correttamente.")
except ImportError:
    raise ImportError(
        "[ERRORE CRITICO] Impossibile trovare 'utils_graphics.py'. "
        "Assicurati che il file sia nella stessa directory di questo notebook su IntelliJ."
    )


# 1. Configurazione dei percorsi dei file
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
base_project_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))

tracking_parquet_path = os.path.join(base_project_dir, "data", "processed", "integrated_tracking_data.parquet")
opta_events_json_path = os.path.join(base_project_dir, "data", "raw", "MA3_opta_matchevent.json")

# Nome cartella di destinazione
NOME_CARTELLA_DESTINAZIONE = "possession02"

# 2. Ricostruzione timeline del possesso da eventi OPTA
print("\n[FASE 1] Parsing e allineamento temporale del flusso eventi JSON...")
if not os.path.exists(opta_events_json_path):
    raise FileNotFoundError(f"[ERRORE] File JSON degli eventi non trovato in: {opta_events_json_path}")

with open(opta_events_json_path, "r", encoding="utf-8") as f:
    eventi_json = json.load(f)

events = eventi_json["liveData"]["event"]

# Codice alfanumerico univoco associato a H per discriminare il possesso
id_h = "3vo5mpj7catp66nrwwqiuhuup"
timeline_possesso = []
parsed_events_for_time = []

for i in range(len(events) - 1):
    ev_attuale = events[i]
    ev_successivo = events[i + 1]

    ts_attuale = int(
        datetime.fromisoformat(
            ev_attuale["timeStamp"].replace("Z", "")
        ).timestamp()
        * 1000
    )
    ts_successivo = int(
        datetime.fromisoformat(
            ev_successivo["timeStamp"].replace("Z", "")
        ).timestamp()
        * 1000
    )

    parsed_events_for_time.append(
        {"period_id": ev_attuale["periodId"], "timestamp_ms": ts_attuale}
    )

    if ev_attuale.get("typeId") == 1 and ev_attuale.get("outcome") == 1:
        squadra_in_possesso = (
            "H" if ev_attuale.get("contestantId") == id_h else "A"
        )
        palla_in_gioco = True
    elif ev_attuale.get("typeId") in [4, 5, 6, 8, 30]:
        palla_in_gioco = False
        squadra_in_possesso = "Nessuno"
    else:
        continue

    timeline_possesso.append(
        {
            "ts_start": ts_attuale,
            "ts_end": ts_successivo,
            "stato_possesso": squadra_in_possesso,
            "palla_attiva": palla_in_gioco,
        }
    )

df_events_time = pd.DataFrame(parsed_events_for_time)
df_timeline = pd.DataFrame(timeline_possesso)
print(f"-> Timeline del possesso ricostruita. Mappati {len(df_timeline)} segmenti fluidi.")

# 3. Segmentazione temporale e normalizzazione spaziale
print("\n[FASE 2] Caricamento del dataset posizionale ad alta frequenza (Parquet)...")
if not os.path.exists(tracking_parquet_path):
    raise FileNotFoundError(f"[ERRORE] File Parquet Tracking non trovato in: {tracking_parquet_path}")

t1_start = df_events_time[df_events_time["period_id"] == 1]["timestamp_ms"].min()
t1_end = df_events_time[df_events_time["period_id"] == 1]["timestamp_ms"].max()
t2_start = df_events_time[df_events_time["period_id"] == 2]["timestamp_ms"].min()
t2_end = df_events_time[df_events_time["period_id"] == 2]["timestamp_ms"].max()

df_tracking = pd.read_parquet(tracking_parquet_path)
df_tracking["periodo"] = 0
df_tracking.loc[
    (df_tracking["timestamp"] >= t1_start)
    & (df_tracking["timestamp"] <= t1_end),
    "periodo",
] = 1
df_tracking.loc[
    (df_tracking["timestamp"] >= t2_start)
    & (df_tracking["timestamp"] <= t2_end),
    "periodo",
] = 2
df_tracking = df_tracking[df_tracking["periodo"].isin([1, 2])].copy()

# Inversione di campo secondo tempo
df_tracking.loc[df_tracking["periodo"] == 2, ["x", "y"]] = -df_tracking.loc[
    df_tracking["periodo"] == 2, ["x", "y"]
]

# Identificazione Portieri
id_portieri = (
    df_tracking.groupby(["team", "player_id"])["x"].mean().reset_index()
)
id_gk_H = (
    id_portieri[id_portieri["team"] == "H"]
    .sort_values(by="x")
    .iloc[0]["player_id"]
)
id_gk_A = (
    id_portieri[id_portieri["team"] == "A"]
    .sort_values(by="x", ascending=False)
    .iloc[0]["player_id"]
)


def prepara_dati_scientifici(df_track):
    X_MIN, X_MAX = -52.5, 52.5
    Y_MIN, Y_MAX = -34.0, 34.0
    PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0
    df_clean = df_track.dropna(subset=["x", "y"]).copy()

    x_clamped = np.clip(df_clean["x"], X_MIN, X_MAX)
    y_clamped = np.clip(df_clean["y"], Y_MIN, Y_MAX)

    df_clean["x_norm"] = ((x_clamped - X_MIN) / (X_MAX - X_MIN)) * PITCH_LENGTH
    df_clean["y_norm"] = ((y_clamped - Y_MIN) / (Y_MAX - Y_MIN)) * PITCH_WIDTH

    is_not_gk = (df_clean["player_id"] != id_gk_H) & (df_clean["player_id"] != id_gk_A)
    centroide_frame = (
        df_clean[is_not_gk]
        .groupby(["timestamp", "team"])[["x_norm", "y_norm"]]
        .mean()
        .reset_index()
    )
    centroide_frame.rename(
        columns={"x_norm": "cx", "y_norm": "cy"}, inplace=True
    )

    df_clean = pd.merge(
        df_clean, centroide_frame, on=["timestamp", "team"], how="left"
    )
    df_clean["x_rel"] = df_clean["x_norm"] - df_clean["cx"]
    df_clean["y_rel"] = df_clean["y_norm"] - df_clean["cy"]
    return df_clean.dropna(subset=["x_rel", "y_rel"])


df_tattico = prepara_dati_scientifici(df_tracking)
df_tattico = df_tattico[
    (df_tattico["player_id"] != id_gk_H) & (df_tattico["player_id"] != id_gk_A)
    ].copy()


# 4. Sinconizzazione del tracking
print("\n[FASE 3] Sincronizzazione ad alta frequenza (Open Play)...")
df_tattico = df_tattico.sort_values("timestamp")
df_timeline = df_timeline.sort_values("ts_start")

df_tattico_filtrato = pd.merge_asof(
    df_tattico,
    df_timeline,
    left_on="timestamp",
    right_on="ts_start",
    direction="backward",
)

# Tiene solo i momenti a palla attiva
df_tattico_filtrato = df_tattico_filtrato[
    df_tattico_filtrato["palla_attiva"] == True
    ].copy()
print(f"-> Righe residue stabili in Open Play: {len(df_tattico_filtrato)}")


# 5. Analisi tattica con il modello addestrato
print("\n[FASE 4] Inizializzazione del classificatore Random Forest per le fasi...")


def individua_modulo_fase_random_forest(df_fase_chunk):
    _modello_ia_temp = globals().get('analista', None)
    if _modello_ia_temp is None or not isinstance(_modello_ia_temp, RandomForestClassifier):
        print("[ERRORE] Il modello 'classificatore_tesi' non è pronto in memoria. Esegui la cella di addestramento!")
        return "N/A", None, None

    # Recupera il modello dallo spazio globale
    modello_ia: RandomForestClassifier = _modello_ia_temp

    # Controllo volumetrico essenziale
    if len(df_fase_chunk) < 150:
        return "N/A", None, None

    # Isolamento dei 10 giocatori dominanti (Filtro anti-bug sostituzioni)
    giocatori_dominanti = (
        df_fase_chunk["player_id"].value_counts().head(10).index
    )
    df_filtrato_10 = df_fase_chunk[
        df_fase_chunk["player_id"].isin(giocatori_dominanti)
    ]

    X_valori = df_filtrato_10[["x_rel", "y_rel"]].values
    if len(X_valori) < 100:
        return "N/A", None, None

    # Calcolo dei 10 centroidi medi del posizionamento
    kmeans_ruoli = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans_ruoli.fit(X_valori)
    centri_ruoli_ideali = kmeans_ruoli.cluster_centers_

    # Ordinamento asse X (profondità) da dietro in avanti
    idx_ordine = np.argsort(centri_ruoli_ideali[:, 0])
    centri_ruoli_ideali = centri_ruoli_ideali[idx_ordine]

    # Vettore di 20 feature per l'IA (10 X + 10 Y)
    vettore_feature = list(centri_ruoli_ideali[:, 0]) + list(
        centri_ruoli_ideali[:, 1]
    )

    # Il Random Forest predice il modulo sulla base delle etichette umane
    modulo_rilevato_ia = modello_ia.predict([vettore_feature])[0]


    try:
        parti_modulo = [int(x) for x in modulo_rilevato_ia.split("-")]
        labels_reparti = []
        for reparto_idx, numero_giocatori in enumerate(parti_modulo):
            labels_reparti.extend([reparto_idx] * numero_giocatori)
        labels_reparti = np.array(labels_reparti)
    except (ValueError, AttributeError):
        labels_reparti = np.zeros(10)

    return modulo_rilevato_ia, centri_ruoli_ideali, labels_reparti


# 6. Pipeline di stampa ogni 15 minuti gioco con differenziazione tra fase offensiva e difensiva
unique_timestamps = sorted(df_tattico_filtrato["timestamp"].unique())
chunks = np.array_split(unique_timestamps, 6)

print("\n=== AVVIO ESTRAZIONE GRAFICA SCIENTIFICA DIVISA PER FASE - RANDOM FOREST ===")

for i, chunk in enumerate(chunks, start=1):
    df_chunk = df_tattico_filtrato[df_tattico_filtrato["timestamp"].isin(chunk)]
    testo_fase = (
        f"Fase di gioco {i} (Minuti indicativi: {int((i-1)*15)}' - {int(i*15)}')"
    )

    print(f"\n" + "=" * 80)
    print(f"[ELABORAZIONE CHUNK CONTESTUALE IA] {testo_fase}")
    print("=" * 80)

    for team_code in ["H", "A"]:
        nome_team = "H (Casa)" if team_code == "H" else "A (Ospite)"
        df_team_chunk = df_chunk[df_chunk["team"] == team_code]

        # Split tra fase offensiva e difensiva
        df_attacco = df_team_chunk[df_team_chunk["stato_possesso"] == team_code]
        df_difesa = df_team_chunk[
            (df_team_chunk["stato_possesso"] != team_code)
            & (df_team_chunk["stato_possesso"] != "Nessuno")
            ]

        # 1. Computazione e disegno della Fase Offensiva (Possesso)
        modulo_att, centri_att, labels_att = (
            individua_modulo_fase_random_forest(df_attacco)
        )
        if modulo_att != "N/A" and centri_att is not None:
            print(f"       -> {nome_team} in FASE DI POSSESSO | Modulo Predetto: {modulo_att}")
            print(f"          [GRAFICA] Esportazione lavagnetta tattica in corso...")
            # Richiamo al modulo grafico
            visualizza_lavagnetta_ungherese2(
                f"{nome_team} - Fase Offensiva",
                team_code,
                testo_fase,
                modulo_att,
                centri_att,
                labels_att,
                nome_script=NOME_CARTELLA_DESTINAZIONE
            )
        else:
            print(f"       [AVVISO] Frame insufficienti in Fase Offensiva per {nome_team} nel chunk {i}.")

        # 2. Computazione e disegno della Fase Difensiva (Non Possesso)
        modulo_dif, centri_dif, labels_dif = (
            individua_modulo_fase_random_forest(df_difesa)
        )
        if modulo_dif != "N/A" and centri_dif is not None:
            print(f"       -> {nome_team} in FASE DI NON POSSESSO | Modulo Predetto: {modulo_dif}")
            print(f"          [GRAFICA] Esportazione lavagnetta tattica in corso...")
            # Richiamo al modulo grafico
            visualizza_lavagnetta_ungherese2(
                f"{nome_team} - Fase Difensiva",
                team_code,
                testo_fase,
                modulo_dif,
                centri_dif,
                labels_dif,
                nome_script=NOME_CARTELLA_DESTINAZIONE
            )
        else:
            print(f"       [AVVISO] Frame insufficienti in Fase Difensiva per {nome_team} nel chunk {i}.")

print("\n=== ANALISI SCIENTIFICA CONTESTUALE AL POSSESSO COMPLETATA CON SUCCESSO ===")

[OK] Modulo 'utils_graphics2' importato correttamente.

[FASE 1] Parsing e allineamento temporale del flusso eventi JSON...
-> Timeline del possesso ricostruita. Mappati 1123 segmenti fluidi.

[FASE 2] Caricamento del dataset posizionale ad alta frequenza (Parquet)...

[FASE 3] Sincronizzazione ad alta frequenza (Open Play)...
-> Righe residue stabili in Open Play: 1440528

[FASE 4] Inizializzazione del classificatore Random Forest per le fasi...

=== AVVIO ESTRAZIONE GRAFICA SCIENTIFICA DIVISA PER FASE - RANDOM FOREST ===

[ELABORAZIONE CHUNK CONTESTUALE IA] Fase di gioco 1 (Minuti indicativi: 0' - 15')
       -> H (Casa) in FASE DI POSSESSO | Modulo Predetto: 2-4-4
          [GRAFICA] Esportazione lavagnetta tattica in corso...
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\MLAnalysis\possession02\H_(Casa)___Fase_Offensiva_Fase_di_gioco_1.png
       -> H (Casa) in FASE DI NON POSSESSO | Modulo Predetto: 4-4-2
   